In [ ]:
# class CompanyMatchResult(BaseModel):
#     is_matched: bool = Field(description="True ถ้าพบว่ามีบริษัทใน company_list False ถ้าไม่เจอหรือไม่มีแน่ใจ")
#     id: int | None = Field(None, description="ID ของบริษัทที่ match จาก company_list")
    
# company_matcher = llm.with_structured_output(CompanyMatchResult)  
# def search_mou_node(state: AgentState) -> Dict[str, Any]:
#     company_name = state.get("company_name")
    
#     if not company_name:
#         return {"context": None}
    
#     company_list = search_company(company_name) # เช่น [{"id": 101, "name": "บริษัท เอ บี ซี จำกัด"}]
    
#     # ถ้าค้นใน DB แล้วไม่พบรายการอะไรเลย
#     if not company_list:
#         return {"context": f"ไม่พบข้อมูลบริษัท {company_name} ในระบบ"}

#     prompt = f"""
#     เปรียบเทียบชื่อบริษัทที่ผู้ใช้ต้องการค้นหา กับ รายชื่อบริษัทที่ค้นพบในระบบ
    
#     ชื่อบริษัทที่ต้องการค้นหา: "{company_name}"
#     รายการบริษัทที่พบในระบบ: {company_list}
    
#     โปรดวิเคราะห์ว่ามีบริษัทในรายการที่ตรงกัน หรือเป็นบริษัทเดียวกันหรือไม่
#     """
    
#     match_result: CompanyMatchResult = company_matcher.invoke(prompt)
    
#     if match_result.is_matched and match_result.matched_id:
#         # นำ ID ที่ AI แมตช์ได้ ไปดึงข้อมูล MOU ต่อ
#         mou_context = get_mou_by_company_id(match_result.matched_id)
#         return {"context": mou_context}
    
#     # ถ้า AI วิเคราะห์แล้วว่าไม่มีชื่อไหนตรงกันเลย
#     return {"context": f"ไม่พบข้อมูล MOU ที่ตรงกับบริษัท {company_name}"}

In [66]:
from ai.services.search import get_person_contact,search_company, search_person
# search_person("บาส")
a = search_company("Bank")

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))
from dotenv import load_dotenv
load_dotenv()
from typing import TypedDict, Optional, Dict, Any
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver
from ai.chains.classify import classify_chain
from schemas.enums import QuestionCategory
from ai.services.search import get_person_contact,search_company, get_mou_status
from core.ai import get_llm
from ai.services.prompt import get_rag_prompt, get_other_prompt

from pydantic import BaseModel, Field

llm = get_llm()

class AgentState(TypedDict):
    question: str
    
    category: QuestionCategory | None
    person_name: str | None
    company_name: str | None
    context: list[Any] | None
    answer: str | None
  
def classify_node(state: AgentState) ->  Dict[str, Any]:
    response = classify_chain.invoke({"question": state["question"]})
    result = response["parsed"].model_dump()
    usage = response["raw"].usage_metadata
    
    # print(f"{result}\n",)
    
    return {
        "category": result["category"],
        "person_name": result["person_name"],
        "company_name": result["company_name"]
    }

def search_contact_node(state: AgentState) -> Dict[str, Any]:
    person_name = state.get("person_name")
    if not person_name:
        return {"context": None}
    
    context = get_person_contact(person_name)
    return {"context": context}


def search_mou_node(state: AgentState) -> Dict[str, Any]:
    company_name = state.get("company_name")
    if not company_name:
        return {"context": None}
    
    company_list  = search_company(company_name)
    context = get_mou_status(company_list)
    return {"context": context}
  
def agent_node(state: AgentState) -> Dict[str, Any]:
    context = state.get("context")
    if context:
        prompt = get_rag_prompt(context, state["question"])
        response = llm.invoke(prompt)
        answer = response.content[0]['text']
        return {"answer": answer}
    
    prompt = get_other_prompt(context, state["question"])
    response = llm.invoke(prompt)
    answer = response.content[0]['text']
    return {"answer": answer}

def route_category(state: AgentState):
    if state.get("category") == QuestionCategory.PERSONAL_CONTACT:
        return "search_contact"
    elif state.get("category") == QuestionCategory.MOU:
        return "mou"
    elif state.get("category") == QuestionCategory.OTHER:
        return "agent"
    
    return "agent"

workflow = StateGraph(AgentState)

workflow.add_node("classify", classify_node)
workflow.add_node("search_contact", search_contact_node)
workflow.add_node("agent", agent_node)
# workflow.add_node("mou", search_mou_node)

workflow.add_edge(START, "classify")
workflow.add_edge("search_contact", "agent")


workflow.add_conditional_edges(
    "classify",
    route_category,
    {
        "search_contact": "search_contact",
        "mou": "mou",
        "other": "agent",
        "agent": "agent"
    }
)
workflow.add_edge("agent", END)

# checkpointer = MemorySaver()
app = workflow.compile()

C:\Users\Pachara Auikim\AppData\Local\Temp\ipykernel_27584\3393269282.py:17: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  async for event in await app.astream_events(initial_state, config=config, version="v3"):
c:\Users\Pachara Auikim\Desktop\Graph_RAG\backend\nextlink-env\Lib\site-packages\langgraph\pregel\main.py:3612: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return AsyncGraphRunStream(graph_aiter, mux)
Direct use of automatic function calling (AFC) in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream. Similarly, direct use of AFC in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message.


{'type': 'event', 'method': 'values', 'params': {'namespace': [], 'timestamp': 1788623755013, 'data': {'question': 'สวัสดี'}, 'interrupts': ()}, 'seq': 1}
{'type': 'event', 'method': 'messages', 'params': {'namespace': [], 'timestamp': 1788623755922, 'data': ({'event': 'message-start', 'role': 'ai', 'id': 'lc_run--01a07248-9708-7451-b984-d54eaf8a8f73', 'metadata': {'provider': 'google_genai'}}, {'ls_integration': 'langchain_chat_model', 'langgraph_step': 1, 'langgraph_node': 'classify', 'langgraph_triggers': ('branch:to:classify',), 'langgraph_path': ('__pregel_pull', 'classify'), 'langgraph_checkpoint_ns': 'classify:1c43eb35-da7e-4d7a-fafa-e971d03b0c1a', 'checkpoint_ns': 'classify:1c43eb35-da7e-4d7a-fafa-e971d03b0c1a', 'ls_provider': 'google_genai', 'ls_model_name': 'gemini-3.5-flash-lite', 'ls_model_type': 'chat', 'ls_temperature': None, 'lc_versions': {'langchain-core': '1.5.4', 'langchain': '1.3.15', 'langchain-google-genai': '4.3.4'}, 'tags': ['map:key:raw'], 'run_id': '01a07248-9

ChatResponse(user_id=0, total_tokens=0, role=<ChatRole.AI: 'ai'>, message='ขออภัย ไม่พบข้อมูลที่เกี่ยวข้องในระบบ', id=1, created_at=datetime.datetime(2026, 9, 5, 22, 55, 55, 933146))

In [ ]:
import logging
from datetime import datetime
from typing import Optional


async def call_agent(user_id: int, question: str) -> Optional[ChatResponse]:
    initial_state = {"question": question}
    answer = None
    total_tokens = 0
    config = {"configurable": {"thread_id": str(user_id)}}

    try:
        async for event in app.astream_events(initial_state, config=config, version="v3"):
            try:
                method = event.get("method")
                params = event.get("params", {})
                data_tuple = params.get("data", ())

                if method == "messages" and isinstance(data_tuple, tuple) and len(data_tuple) >= 2:
                    payload = data_tuple[0] if isinstance(data_tuple[0], dict) else {}
                    metadata = data_tuple[1] if isinstance(data_tuple[1], dict) else {}

                    node_name = metadata.get("langgraph_node", "unknown_node")
                    event_type = payload.get("event")

                    if event_type == "message-start":
                        node_label = node_mapping.get(node_name, node_name)
                        print(f"Starting node: {node_label}")

                    elif event_type == "message-finish":
                        total_tokens += payload.get("total_tokens", 0)

                        if node_name == "classify":
                            print("...กำลังค้นหาข้อมูล")

                elif method == "values":
                    state_data = params.get("data", {})
                    if isinstance(state_data, dict) and "answer" in state_data:
                        answer = state_data["answer"]

            except Exception as e:
                continue

    except Exception as e:
        answer = "เกิดข้อผิดพลาดในการประมวลผล กรุณาลองใหม่อีกครั้ง"

     
    if not answer:
        answer = "เกิดข้อผิดพลาดในการประมวลผล กรุณาลองใหม่อีกครั้ง"
    try:
            params = {
                "user_id": user_id,
                "total_tokens": total_tokens,
                "message": answer,
                "role": "ai"
            }
            
            # Validate ด้วย Pydantic
            response_dto = ChatResponseCreate(**params)

            # TODO: Insert Into DB 
            # await db.save_chat(response_dto)
            
            return ChatResponse(id=1, created_at=datetime.now(), **params)
        
    except Exception as e:
            return None

In [41]:
a = await call_agent(0, "สวีดัส")
print(a)

...กำลังวิเคราะห์คำถาม
...กำลังค้นหาช้อมูล
ไม่เจอข้อมูล
ขออภัย ไม่พบข้อมูลที่เกี่ยวข้องในระบบ


In [ ]:
from IPython.display import Image, display

display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:


res = llm.invoke(get_other_prompt("สวีดัสครับ"))
print(res.content[0]['text'])

สวีดัสค่ะคุณผู้ใช้ (แอบยิ้ม) 

ขวัญใจยินดีต้อนรับนะคะ วันนี้มีเรื่องอะไรให้ขวัญใจช่วยเหลือเป็นพิเศษไหมเอ่ย หรือถ้าหากต้องการสอบถามเกี่ยวกับข้อมูลการติดต่อบุคคล, ข้อตกลงความร่วมมือ (MOU), ความสัมพันธ์ระหว่างองค์กร, หรือกิจกรรมของบริษัท แจ้งขวัญใจได้เลยนะคะ ขวัญใจพร้อมให้บริการค่ะ 😊
